# DEAI-opdrachten Classification


waarom classification? 
- category voorspellen in plaats van numerieke zoals regessie
- heeft wel een target nodig

imports

In [31]:
#df 
import pandas as pd

#data splitsen in train/test sets
from sklearn.model_selection import train_test_split

#beslisboom  te kunnen classificiren 
from sklearn.tree import DecisionTreeClassifier

 # evaluatiemetrics voor modelprestaties
 # Accuracy: de procentuele mate waarin het model goede voorspellingen heeft gemaakt. => hoe vaak het model gelijk heeft

# Recall: de procentuele mate waarin het model échte positieve gevallen als “positief” classificeert. => van alle echte positieve gevallen, hoeveel worden correct gevonden

# Precision: welk deel van de als positief voorspelde gevallen ook écht positief is. => # => hoeveel zijn echt echt correct, en niet false positives
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# f1 : harmonisch gemiddelde van precision en recall => Combineert beide metrics in één score => balans tussen fout-positieven en fout-negatieven

df = pd.read_excel("AmesHousing.xlsx", sheet_name="AmesHousing")

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ID             2930 non-null   int64  
 1   SalePrice      2930 non-null   int64  
 2   Garage         2930 non-null   str    
 3   Overall Qual   2930 non-null   int64  
 4   Gr Liv Area    2930 non-null   int64  
 5   Total Bsmt SF  2929 non-null   float64
 6   Lot Area       2930 non-null   int64  
 7   Year Built     2930 non-null   int64  
 8   Full Bath      2930 non-null   int64  
 9   Bedroom AbvGr  2930 non-null   int64  
 10  Neighborhood   2930 non-null   str    
 11  House Style    2930 non-null   str    
dtypes: float64(1), int64(8), str(3)
memory usage: 274.8 KB


In [32]:

# omzetten naar numerieke waarde zodat het model ermee kan werken
df["HasGarage"] = (df["Garage"] == "yes").astype(int)
target = "HasGarage"


In [33]:
#logboek
logboek = pd.DataFrame(columns=[
    "Model",
    "Experiment",
    "Features",
    "Hyperparameters",
    "Expected effect",
    "Result observation",
    "Conclusion",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score"
])



In [ ]:

#exp 1 model 1 => garage voorspellen 
#features => helpt bij voorspelling
features = ["Overall Qual", "Gr Liv Area"]

#category
cat_cols = ["Neighborhood"]

# originele dataset kopiëren zodat we de originele data niet aanpassen en missende numerieke waarden vervangen door de mediaan
df_exp = df.copy()
df_exp = df_exp.fillna(df_exp.median(numeric_only=True))

# categorische variabelen omzetten naar dummy variabelen (one-hot encoding)
# drop_first=True voorkomt multicollineariteit (1 categorie wordt weggelaten)
df_exp = pd.get_dummies(df_exp, columns=cat_cols, drop_first=True)

#colomn ophalen
encoded_cols = [col for col in df_exp.columns if "Neighborhood" in col]

# input features
X = df_exp[features + encoded_cols]
# target = wat we willen voorspellen
y = df_exp[target]

# dataset splitsen in training (80%) en testdata (20%)
# random_state zorgt ervoor dat de split altijd hetzelfde is
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# beslisingboom initialiseren
# max_depth = maximale diepte van de boom (voorkomt overfitting)
# min_samples_split = minimum aantal samples nodig om te splitsen
model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    random_state=42
)

# model trainen op de trainingsdata
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# percentage correcte voorspellingen => aantal correcte voorspelling van totaal aantal voorspellingen  
accuracy = accuracy_score(y_test, y_pred)

# van alle voorspelde positieve gevallen, hoeveel zijn correct
precision = precision_score(y_test, y_pred)

# van alle echte positieve gevallen, hoeveel zijn gevonden => bijv 10 hebben een garage, maar hoeveel heeft model gevonden? 
recall = recall_score(y_test, y_pred)

# balans tussen precision en recall
f1 = f1_score(y_test, y_pred)

print(accuracy, precision, recall, f1)

logboek.loc[len(logboek)] = {
    "Model": "Garage",
    "Experiment": "1",
    "Features": "Overall Qual, Gr Liv Area, Neighborhood",
    "Hyperparameters": "max_depth=5, min_samples_split=10",
    "Expected effect": "Sterke basisfeatures (grootte + kwaliteit + locatie) zouden een stabiel baseline model moeten geven",
    "Result observation": "Het model werkt al heel goed en vindt bijna alle garages.",
    "Conclusion": " Ik heb een simpel model gemaakt met basisgegevens zoals grootte, kwaliteit en buurt.",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}



0.9402730375426621 0.9524647887323944 0.9854280510018215 0.9686660698299016


In [ ]:
#exp 2 model 1
#features, gebruikt om voorspellingen te maken
features = ["Overall Qual", "Gr Liv Area", "Year Built", "Full Bath"]
#category => niet numeriek 
cat_cols = ["Neighborhood"]

#zorg dat echte data niet mee verandert
df_exp = df.copy()
#data die ontbreekt 
df_exp = df_exp.fillna(df_exp.median(numeric_only=True))

#one hot encoding => convert category naar numerieke => belangrijk voor beslisingboom 
df_exp = pd.get_dummies(df_exp, columns=cat_cols, drop_first=True)
#alle neibourhood
encoded_cols = [col for col in df_exp.columns if "Neighborhood" in col]

#features => wat model gebruikt om voorspellingen te maken, 
X = df_exp[features + encoded_cols]
#target => voorspeling => heeft targer wel of geen garage?
y = df_exp[target]


#test data voorbereiden
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#beslisingboom
#max depth => hoe diep boom kan groeien =>  hoeveel niveau/splitsing =>overfitting/ under fitting
# elke leaf moet minste n bevatten
# random state blift dezelfde => dezelfde gedrag elke keer
model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)

# model trainen op de trainingsdata
model.fit(X_train, y_train)
#voorspelling data 
y_pred = model.predict(X_test)

#
# percentage correcte voorspellingen
accuracy = accuracy_score(y_test, y_pred)

#of data echt echt klopt 
precision = precision_score(y_test, y_pred)

#  de procentuele mate waarin het model échte positieve gevallen als “positief” classificeert.
#false negative 
recall = recall_score(y_test, y_pred)

# balans tussen precision en recall => alleen hoog als beide hoog is 
f1 = f1_score(y_test, y_pred)


print(accuracy, precision, recall, f1)

logboek.loc[len(logboek)] = {
    "Model": "Garage",
    "Experiment": "2",
    "Features": "Overall Qual, Gr Liv Area, Year Built, Full Bath",
    "Hyperparameters": "max_depth=10, min_samples_leaf=5",
    "Expected effect": "Ik heb extra info toegevoegd zoals bouwjaar en aantal badkamers.",
    "Result observation": "Meer data helpt een beetje, maar niet altijd veel.",
    "Conclusion": "",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}



0.9334470989761092 0.9473684210526315 0.9836065573770492 0.9651474530831099


In [36]:
#exp 3, model 1
#features, helpt bij voorspellingen
features = ["Overall Qual", "Gr Liv Area", "Lot Area", "Bedroom AbvGr"]
#col => niet numerieke waarden
cat_cols = ["Neighborhood"]

#zorgt dat data niet mee verandert
df_exp = df.copy()
#data die ontbrekt => naar median gezet
df_exp = df_exp.fillna(df_exp.median(numeric_only=True))

#one hot encoding => convert category naar numerieke => belangrijk voor beslisingboom
df_exp = pd.get_dummies(df_exp, columns=cat_cols, drop_first=True)

encoded_cols = [col for col in df_exp.columns if "Neighborhood" in col]

X = df_exp[features + encoded_cols]
y = df_exp[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(
    max_depth=None,
    min_samples_split=2,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("EXP 3 - OVERFITTING TEST")
print(accuracy, precision, recall, f1)

logboek.loc[len(logboek)] = {
    "Model": "Garage",
    "Experiment": "3",
    "Features": "Overall Qual, Gr Liv Area, Lot Area, Bedroom AbvGr",
    "Hyperparameters": "max_depth=None",
    "Expected effect": "Zeer diep model kan overfitten en trainingprestatie verhogen",
    "Result observation": "Testprestatie blijft stabiel of verslechtert licht",
    "Conclusion": "Model vertoont lichte overfitting en extra complexiteit levert weinig winst op",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}



EXP 3 - OVERFITTING TEST
0.9129692832764505 0.9527272727272728 0.9544626593806922 0.9535941765241128


In [37]:
#model 2 exp 1

#moedl 2 target 
target = "Overall Qual"

# features = de input variabelen die het model gebruikt om te leren en voorspellen
# deze kenmerken hebben invloed op de kans op een garage
features = ["Gr Liv Area", "Year Built", "Full Bath"]
# cat_cols = categorische (niet-numerieke) kolommen
# deze moeten eerst worden omgezet naar cijfers (one-hot encoding)
cat_cols = ["Neighborhood"]

#maakt een kopie van de dataset
# zodat de originele data niet per ongeluk verandert
df_exp = df.copy()

# missing values worden vervangen
# dit voorkomt fouten in het model en behoudt realistische waarden
df_exp = df_exp.fillna(df_exp.median(numeric_only=True))

# one-hot encoding:
# zet tekstcategorieën (zoals Neighborhood) om naar 0/1 kolommen
# zodat het model ze kan begrijpen (beslisbomen werken alleen met cijfers)
df_exp = pd.get_dummies(df_exp, columns=cat_cols, drop_first=True)

# haal alle nieuw gemaakte Neighborhood kolommen op
# zodat ze meegenomen worden als input features
encoded_cols = [col for col in df_exp.columns if "Neighborhood" in col]

# input data/features 
# combineert numerieke features + encoded categorische features
X = df_exp[features + encoded_cols]

# y = target (wat je wilt voorspellen)
y = df_exp[target]

# splitst data in training en test set
# 80% training, 20% testen 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# beslising boom 
# max_depth=None, geen limiet op diepte, kan heel complex worden
# min_samples_split=2, minimale data nodig om een split te maken
# random_state zorgt voor dezelfde resultaten
model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print(accuracy, precision, recall, f1)

logboek.loc[len(logboek)] = {
    "Model": "Quality",
    "Experiment": "4",
    "Features": "Gr Liv Area, Year Built, Full Bath, Neighborhood",
    "Hyperparameters": "max_depth=5, min_samples_split=10",
    "Expected effect": "Basisfeatures zouden algemene woningkwaliteit goed moeten schatten",
    "Result observation": "Goede algemene voorspellingen maar moeite met extreme kwaliteitsklassen",
    "Conclusion": "model is stabiel maar mist precisie voor fijnere klassen",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}


0.537542662116041 0.5308124769852244 0.537542662116041 0.48475850492293615


c:\Users\catsg\Desktop\DEAI_portfolio\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [38]:
#model 2 exp 2

#features = de input variabelen die het model gebruikt om te leren en voorspellen
# deze kenmerken hebben invloed op de kans op een Quality
features = ["Gr Liv Area", "Year Built", "Full Bath", "Lot Area"]

df_exp = df.copy()
df_exp = df_exp.fillna(df_exp.median(numeric_only=True))

df_exp = pd.get_dummies(df_exp, columns=["Neighborhood"], drop_first=True)

encoded_cols = [col for col in df_exp.columns if "Neighborhood" in col]

X = df_exp[features + encoded_cols]
y = df_exp[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print(accuracy, precision, recall, f1)

logboek.loc[len(logboek)] = {
    "Model": "Quality",
    "Experiment": "5",
    "Features": "Gr Liv Area, Year Built, Full Bath, Lot Area",
    "Hyperparameters": "max_depth=10, min_samples_leaf=5",
    "Expected effect": "Meer structurele informatie moet betere scheiding tussen kwaliteitsniveaus geven",
    "Result observation": "Verbetering in weighted metrics, vooral recall",
    "Conclusion": "Extra features helpen, maar verbetering is beperkt door datasetcomplexiteit",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}



0.5358361774744027 0.5321960951914746 0.5358361774744027 0.524349763581459


In [39]:
#model 2 exp 3
features = ["Gr Liv Area", "Year Built", "Lot Area", "Bedroom AbvGr"]

df_exp = df.copy()
df_exp = df_exp.fillna(df_exp.median(numeric_only=True))

df_exp = pd.get_dummies(df_exp, columns=["Neighborhood"], drop_first=True)

encoded_cols = [col for col in df_exp.columns if "Neighborhood" in col]

X = df_exp[features + encoded_cols]
y = df_exp[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = DecisionTreeClassifier(
    max_depth=None,
    min_samples_split=2,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

logboek.loc[len(logboek)] = {
    "Model": "Quality",
    "Experiment": "6",
    "Features": "Gr Liv Area, Year Built, Lot Area, Bedroom AbvGr",
    "Hyperparameters": "max_depth=None",
    "Expected effect": "Zeer flexibel model kan trainingsdata overfitten",
    "Result observation": "Prestatie op testdata wordt minder stabiel",
    "Conclusion": "Overfitting zichtbaar; model generaliseert slechter bij te hoge complexiteit",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}



In [40]:
logboek.to_excel("logboek_classificatie1.xlsx", index=False)

print("Logboek opgeslagen!")


Logboek opgeslagen!
